# Tutorial 6: Creating multi-feeder networks and integrating transmission level

Over the last tutorials, we focused on creating a distribution grid at the grid level with a single feeder.

While this is sufficient for many test cases, the bayesgrid tool can generate grids with multiple feeders and different voltage levels.

Additionally, bayesgrid also accounts for functions that incorporate transmission level into the distribution grid.

We will cover both of these aspects in this tutorial.



# Importing required modules

bayesgrid have a dedicated set of functions for multi feeder, under multi_feeder_grid_generator. Let's import those now.

In [3]:
from bayesgrid import *
import osmnx as ox
import numpy as np
import pandas as pd
import pandapower as pp
import pandapower.topology


# Generating a grid with multiple feeders

To generate a grid with multiple feeders, we use the "create_multi_feeder_network" function.

We can use the 4 accepted query formats from bayesgrid (see tutorial 2 for more information)

We have one optional keyword: transmission_connected. If set to True, it will generate the distribution grid coupled with transmission level elements (transformers, lines, generators)

First we will focus on multiple feeder integration, so let's set it to False for now.


The function have 4 output:
* net: the pandapower network with multiple feeders. 
* G_forest: the forest graph (a collection of multiple trees, one for each feeder). This can be used for graph-related operations within networkx.
* sub_nodes: the nodes that are substations within the generated grid
* single_feeder_list: a list of every single feeder network (voronoi partition) within the multiple feeder grid. you may use each of the individual networks in any of the previous tutorial to generate the network parameters.


In [ ]:


# A point in Santo Amaro, São Paulo
query_point = (-23.649, -46.702) 


query_type = 'point'
radius = 1500

net, G_forest, sub_nodes, single_feeder_list = create_multi_feeder_network(query_point, query_type='point', dist=radius,
                                                                           transmission_connected=False)



# Voltage levels

Bayesgrid have a dedicated function for voltage level. This function will create all required transformers for multiple voltage levels within the grid

You may define the low voltage level (in kV). Otherwise it will set to 127V (default for the Brazilian grid)

In [ ]:

net = apply_voltage_levels(net, G_forest, sub_nodes, lv_kv=0.220)

# Visualization

Bayesgrid have dedicated functions for the visualization of the generated grids.

It will use the objects generated by the previous functions.

The "plot_multi_feeder_graph" will focus on drawing the voronoi partitions that define the customers served for each feeder:


In [ ]:
fig, ax = plot_multi_feeder_graph(G_forest, sub_nodes)



On the other hand, the "plot_detailed_integrated_system" will focus on detailing the specific components of the grid: transformers, voltage level, and grid sources.

You may choose a minimum voltage level (in kV) to draw transformers (min_kv parameter), so you can focus on high/medium/low voltage representations

You may also remove the transformers from the figure with the draw_transformers parameter.

In [ ]:
plot_detailed_integrated_system(net,draw_transformers=True,min_kv=0)

In [ ]:
plot_detailed_integrated_system(net,draw_transformers=False)

## Incorporating transmission and generation

Bayesgrid have a dedicated function for creating transmission and generation level components within an existing network (pandapower object): "build_transmission_generation_network


You may choose a distinct query type and parameters for the transmission/generation level. For example, you may generate a distribution level for a smaller area but generate transmission/generation level for a bigger one.


In [ ]:

net, full_graph = build_transmission_generation_network(net, dist_radius=6000,line_filter='["power"~"line"]',
                                                        query_point=query_point)


The previous defined functions can also be used for visulization:

In [ ]:
plot_detailed_integrated_system(net,draw_transformers=True,min_kv=1,to_save=False)
plt.show()

# Generating bayesian models

Finally, for the most important part. bayesgrid have a dedicated function to run the bayesian hierarchical models and generate synthetic samples within multifeeder networks.

Run the "generate_and_save_full_system_data" function. 

Choose an output prefix for the files.

Every file will be generated within the folder you are running the code. As 4 .csv files (one for each BHM).

In [ ]:

generate_and_save_full_system_data(net, n_samples=1000, output_prefix="santo_andre")

# Exporting to opendss

bayesgrid also have a dedicated function to export the generated synthetic grids (and fixed topology) to opendss format.

The pandapower object for the grid's topology, coupled with the path for the .csv files of the generated samples, are both required.

Each sample of the bayesian models will generate a different .dss files. All .dss files will be saved under the "OPENDSS_EXPORT_FOLDER" path.

In [ ]:
prefix_city = 'santo_andre'

path_power = f'{prefix_city}_bus_power_and_phase_SAMPLES.csv'
path_freq = f'{prefix_city}_bus_frequency_SAMPLES.csv'
path_dur = f'{prefix_city}_bus_duration_SAMPLES.csv'
path_imp = f'{prefix_city}_line_impedance_SAMPLES.csv'

In [ ]:
OPENDSS_EXPORT_FOLDER = 'multi_feeder_synthetic_net_opendss'

In [ ]:
from BayesGrid.src.bayesgrid.exporters import save_synthetic_network

In [ ]:
save_synthetic_network(
    base_net=net,
    path_power_phase=path_power,
    path_frequency=path_freq,
    path_duration=path_dur,
    path_impedance=path_imp,
    output_path=OPENDSS_EXPORT_FOLDER,
    format='opendss'
)

# Worldwide bigger systems

In [ ]:
# A point in Santo Amaro, São Paulo
# santo andre (smaller city)
# query_point = (-23.649, -46.702) 
# prefix_city = 'small_santo_andre'
# radius = 300
# query_type = 'point'

# santo amaro
# query_point = (-23.649, -46.702) 
# prefix_city = 'santo_amaro'
# query_type = 'point'
# radius = 3000 # new york

# los angeles
query_point = (34.05,-118.24)
prefix_city = 'los_angeles'
query_type = 'point'
radius = 2000 # new york

# madrid
# query_point = (40.4168,-3.7038)
# prefix_city = 'madrid'
# query_type = 'point'
# radius = 5000 # new york

# new york 
# query_point = (40.7127,-74.0059)
# prefix_city = 'new_york'
# query_type = 'point'
# radius = 5000

# tokyo
# query_point = (35.652832,139.839478)
# prefix_city = 'tokyo'
# query_type = 'point'
# radius = 5000

net, G_forest, sub_nodes, single_feeder_list = create_multi_feeder_network(query_point, query_type, dist=radius,
                                                                           transmission_connected=True)


net = apply_voltage_levels(net, G_forest, sub_nodes, lv_kv=0.220)

In [ ]:
fig, ax = plot_multi_feeder_graph(G_forest, sub_nodes)


# plt.savefig(f"{prefix_city}_voronoi.svg", format='svg', dpi=100)
net

In [ ]:
plot_detailed_integrated_system(net,draw_transformers=False,draw_ext_grids=True,min_kv=0,show_legend=False,to_save=True)


In [ ]:

net, full_graph = build_transmission_generation_network(net, dist_radius=20000,line_filter='["power"~"line"]')

In [ ]:
generate_and_save_full_system_data(net, n_samples=1000, output_prefix=prefix_city)


In [ ]:
import networkx as nx
import pandapower.topology as ppt

# 1. Create the NX graph from the pandapower net
graph_from_net = ppt.create_nxgraph(net)

## Saving to opendss

In [ ]:
import os

OPENDSS_EXPORT_FOLDER = f'{prefix_city}_synthetic_net_opendss'


path_power = os.path.join(OPENDSS_EXPORT_FOLDER, f'{prefix_city}_bus_power_and_phase_SAMPLES.csv')
path_freq = os.path.join(OPENDSS_EXPORT_FOLDER, f'{prefix_city}_bus_frequency_SAMPLES.csv')
path_dur = os.path.join(OPENDSS_EXPORT_FOLDER, f'{prefix_city}_bus_duration_SAMPLES.csv')
path_imp = os.path.join(OPENDSS_EXPORT_FOLDER, f'{prefix_city}_line_impedance_SAMPLES.csv')

from BayesGrid.src.bayesgrid.exporters import save_synthetic_network


save_synthetic_network(
    base_net=net,
    path_power_phase=path_power,
    path_frequency=path_freq,
    path_duration=path_dur,
    path_impedance=path_imp,
    output_path=OPENDSS_EXPORT_FOLDER,
    format='opendss'
)

## Visualization of a single sample


In [ ]:
# Choose which sample to visualize (0 to N_SAMPLES - 1)
OPENDSS_EXPORT_FOLDER = f'{prefix_city}_synthetic_net_opendss'


path_power = os.path.join(OPENDSS_EXPORT_FOLDER, f'{prefix_city}_bus_power_and_phase_SAMPLES.csv')
path_freq = os.path.join(OPENDSS_EXPORT_FOLDER, f'{prefix_city}_bus_frequency_SAMPLES.csv')
path_dur = os.path.join(OPENDSS_EXPORT_FOLDER, f'{prefix_city}_bus_duration_SAMPLES.csv')
path_imp = os.path.join(OPENDSS_EXPORT_FOLDER, f'{prefix_city}_line_impedance_SAMPLES.csv')

df_power = pd.read_csv(path_power)
df_freq = pd.read_csv(path_freq)
df_dur = pd.read_csv(path_dur)
df_imp = pd.read_csv(path_imp)





In [ ]:
def get_bus_geo_dict(net):
    bus_geo = {}
    for i in net.bus.index:
        try:
            # Parse GeoJSON: {"coordinates": [lon, lat], ...}
            coords = json.loads(net.bus.at[i, 'geo'])['coordinates']
            bus_geo[i] = (coords[0], coords[1]) 
        except: pass
    return bus_geo
source_bus = net.ext_grid.bus.iloc[0] 

bus_geo_data = get_bus_geo_dict(net)

bus_geo_data = get_bus_geo_dict(net)

In [ ]:
# Filter DataFrames
SAMPLE_ID = 800
print(f"Filtering data for Sample ID: {SAMPLE_ID}")
sample_power = df_power[df_power['sample_id'] == SAMPLE_ID].set_index('bus_id')
sample_freq = df_freq[df_freq['sample_id'] == SAMPLE_ID].set_index('bus_id')
sample_dur = df_dur[df_dur['sample_id'] == SAMPLE_ID].set_index('bus_id')
sample_imp = df_imp[df_imp['sample_id'] == SAMPLE_ID].set_index('line_id')


In [ ]:
from matplotlib.lines import Line2D
from matplotlib.patches import Circle
from matplotlib.ticker import FormatStrFormatter

source_bus_list = net.ext_grid.bus.values
# --- 1. Map Phases to Colors ---
# Define standard colors for phases
phase_colors = {
    'A': 'red', 'B': 'green', 'C': 'blue',
    'AB': 'yellow', 'BC': 'cyan', 'CA': 'magenta', # Note: using AC for consistency
    'ABC': 'white', 'nan': 'black'
}

node_colors = []
for node in graph_from_net.nodes:
    if node in source_bus_list:
        node_colors.append('black') # Source is black
    else:
        try:
            phase = str(sample_power.loc[node, 'phase'])
            # Handle potential 'CA' vs 'AC' naming differences

            node_colors.append(phase_colors.get(phase, 'black'))
        except Exception as e:
            print(e)
            node_colors.append('black')

# --- 2. Plotting ---
plt.figure(figsize=(8, 8))

# Draw edges
nx.draw_networkx_edges(graph_from_net, pos=bus_geo_data, edge_color='black', alpha=0.5)

# Draw nodes
nx.draw_networkx_nodes(
    graph_from_net, pos=bus_geo_data,
    node_size=[200 if n in source_bus_list else 10 for n in graph_from_net.nodes],
    node_color=node_colors,
)

# Create Custom Legend
handles = [
    Circle(0, color='red', label='A'),
    Circle(0, color='green', label='B'),
    Circle(0, color='blue', label='C'),
    Circle(0, color='yellow', label='AB'),
    Circle(0, color='cyan', label='BC'),
    Circle(0, color='magenta', label='AC'),
    Circle(0, color='white', label='ABC'),
    Circle(0, color='black', label='Source')
]
plt.legend(handles=handles, loc='best', title="Phases",ncols=10,framealpha=1)
#plt.title(f"Network Phase Allocation (Sample {SAMPLE_ID})")
plt.axis('off')
plt.show()

In [ ]:
# --- 1. Prepare Data ---
# Extract power values for each phase, aligned with graph nodes
p_a_values = [sample_power.loc[n, 'P_A'] if n in sample_power.index else 0 
              for n in graph_from_net.nodes]
p_b_values = [sample_power.loc[n, 'P_B'] if n in sample_power.index else 0 
              for n in graph_from_net.nodes]
p_c_values = [sample_power.loc[n, 'P_C'] if n in sample_power.index else 0 
              for n in graph_from_net.nodes]

# --- 2. Calculate Shared Limits ---
# We need a single min/max for all 3 plots to make colors comparable
all_p_values = p_a_values + p_b_values + p_c_values
p_min = min(all_p_values)
p_max = max(all_p_values)

print(f"Power Range: {p_min:.2f} kW to {p_max:.2f} kW")

# --- 3. Plotting ---
fig, axes = plt.subplots(1, 3, figsize=(16, 6))

# Helper list for looping
phases_data = [
    ('Phase A', p_a_values, axes[0]),
    ('Phase B', p_b_values, axes[1]),
    ('Phase C', p_c_values, axes[2])
]

for phase_name, p_values, ax in phases_data:
    # Draw faint edges for context
    nx.draw_networkx_edges(graph_from_net, pos=bus_geo_data, ax=ax, 
                           edge_color='gray', alpha=0.3)
    
    # Draw nodes colored by power
    nodes = nx.draw_networkx_nodes(
        graph_from_net, pos=bus_geo_data, ax=ax,
        node_size=[150 if n==source_bus else 60 for n in graph_from_net.nodes],
        node_color=p_values,
        cmap=plt.cm.inferno, # 'inferno' is great for intensity (black->red->yellow)
        vmin=p_min, vmax=p_max, # <--- Shared scale
        edgecolors='black'
    )
    ax.set_title(f"{phase_name}", fontsize=16)
    ax.axis('off')

# --- 4. Add Shared Colorbar ---
# Create a dummy mappable for the colorbar
sm = plt.cm.ScalarMappable(cmap=plt.cm.inferno, norm=plt.Normalize(vmin=p_min, vmax=p_max))
sm.set_array([])

# Place colorbar at the bottom
cbar = fig.colorbar(sm, ax=axes.ravel().tolist(), shrink=0.6, location='bottom', pad=-0.05)
cbar.set_label('Active Power (kW)', fontsize=14)

plt.show()

In [ ]:


# Prepare Power Arrays (for columns 2-4)
p_a_values = [sample_power.loc[n, 'P_A'] if n in sample_power.index else 0 for n in graph_from_net.nodes]
p_b_values = [sample_power.loc[n, 'P_B'] if n in sample_power.index else 0 for n in graph_from_net.nodes]
p_c_values = [sample_power.loc[n, 'P_C'] if n in sample_power.index else 0 for n in graph_from_net.nodes]

# Calculate shared limits for power plots
all_p_values = p_a_values + p_b_values + p_c_values
p_min = min(all_p_values)
p_max = max(all_p_values)

# --- 2. Prepare Colors for Phases (Column 1) ---
# UPDATED color mapping
phase_colors = {
    'A': 'red', 'B': 'green', 'C': 'blue',
    'AB': 'yellow', 'BC': 'cyan', 'AC': 'magenta', # Note: using AC for consistency
    'ABC': 'white', 'nan': 'black'
}

node_colors_phase = []
for node in graph_from_net.nodes:
    if node == source_bus:
        node_colors_phase.append('black') # Source is black
    else:
        try:
            phase = str(sample_power.loc[node, 'phase'])
            if phase == 'CA': phase = 'AC'
            node_colors_phase.append(phase_colors.get(phase, 'black'))
        except KeyError:
            node_colors_phase.append('black')

# --- 3. Setup Plot ---
fig, axes = plt.subplots(1, 4, figsize=(24, 6))

# Common drawing settings
edge_color = 'gray'
edge_alpha = 0.3
node_size_source = 150
node_size_regular = 50

# ====================================================================
# --- Plot 1: Phase Allocation ---
# ====================================================================
ax = axes[0]
nx.draw_networkx_edges(graph_from_net, pos=bus_geo_data, ax=ax, edge_color=edge_color, alpha=edge_alpha)
nx.draw_networkx_nodes(
    graph_from_net, pos=bus_geo_data, ax=ax,
    node_size=[node_size_source if n==source_bus else node_size_regular for n in graph_from_net.nodes],
    node_color=node_colors_phase,
    edgecolors='black'
)
ax.set_title("Phase Allocation", fontsize=16)
ax.axis('off')

# --- Custom Legend Below Plot 1 ---
handles = [Circle(0, color=c, label=l) for l, c in phase_colors.items() if l != 'nan']
# Add Source manually if desired, or assume 'nan' covers it
handles.append(Circle(0, color='black', label='Feeder'))

# bbox_to_anchor moves the legend outside the plot. (0.5, -0.05) is centered below.
ax.legend(
    handles=handles, 
    loc='upper center', 
    bbox_to_anchor=(0.5, -0.02), 
    ncol=3, 
    fontsize=14, 
    frameon=False,
)

# ====================================================================
# --- Plots 2, 3, 4: Power Demand ---
# ====================================================================
power_plots = [
    ('Power - Phase A', p_a_values, axes[1]),
    ('Power - Phase B', p_b_values, axes[2]),
    ('Power - Phase C', p_c_values, axes[3])
]

for title, p_vals, ax in power_plots:
    nx.draw_networkx_edges(graph_from_net, pos=bus_geo_data, ax=ax, edge_color=edge_color, alpha=edge_alpha)
    nx.draw_networkx_nodes(
        graph_from_net, pos=bus_geo_data, ax=ax,
        node_size=[node_size_source if n==source_bus else node_size_regular for n in graph_from_net.nodes],
        node_color=p_vals,
        cmap=plt.cm.inferno,
        vmin=p_min, vmax=p_max,
        edgecolors='black'
    )
    ax.set_title(title, fontsize=16)
    ax.axis('off')

# ====================================================================
# --- Shared Colorbar for Power (Below Plots 2-4) ---
# ====================================================================
sm = plt.cm.ScalarMappable(cmap=plt.cm.inferno, norm=plt.Normalize(vmin=p_min, vmax=p_max))
sm.set_array([])

# # Add colorbar associated with the last 3 axes
cbar = fig.colorbar(sm, ax=axes, shrink=0.6, location='bottom', pad=0.05)
cbar.set_label('Active Power (kW)', fontsize=14)

plt.show()

In [ ]:
# --- Prepare Data for Plotting ---
# Map the values to the graph nodes (ensuring order matches graph.nodes)
freq_values = [sample_freq.loc[n, 'CAIFI_FIC'] if n in sample_freq.index else 0 
               for n in graph_from_net.nodes]

dur_values = [sample_dur.loc[n, 'CAIDI_DIC'] if n in sample_dur.index else 0 
              for n in graph_from_net.nodes]

# --- Plotting ---
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: Frequency (CAIFI)
ax1 = axes[0]
nodes1 = nx.draw_networkx_nodes(
    graph_from_net, pos=bus_geo_data, ax=ax1,
    node_size=60, node_color=freq_values,
    cmap=plt.cm.YlOrRd, edgecolors='black'
)
nx.draw_networkx_edges(graph_from_net, pos=bus_geo_data, ax=ax1, edge_color='black', alpha=0.4)
cbar1 = plt.colorbar(nodes1, ax=ax1)
cbar1.set_label('CAIFI (Failures/Year)', rotation=270, labelpad=20, fontsize=20)
cbar1.ax.tick_params(labelsize=18) # Adjust labelsize as desired
#ax1.set_title("Predicted Failure Frequency (CAIFI)")
ax1.axis('off')

# Plot 2: Duration (CAIDI)
ax2 = axes[1]
nodes2 = nx.draw_networkx_nodes(
    graph_from_net, pos=bus_geo_data, ax=ax2,
    node_size=60, node_color=dur_values,
    cmap=plt.cm.YlOrRd, edgecolors='black'
)
nx.draw_networkx_edges(graph_from_net, pos=bus_geo_data, ax=ax2, edge_color='black', alpha=0.4)
cbar2 = plt.colorbar(nodes2, ax=ax2)
cbar2.set_label('CAIDI (Hours/Interruption)', rotation=270, labelpad=20, fontsize=20)
cbar2.ax.tick_params(labelsize=18) # Adjust labelsize as desired
#ax2.set_title("Predicted Failure Duration (CAIDI)")
ax2.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# --- 1. Prepare Edge Widths (Based on Phases) ---
# Helper to count phases
def get_phase_count(phase_str):
    if pd.isna(phase_str): return 1
    return len(str(phase_str).replace('ABC', '3')) 

phase_count_to_width = {1: 4.0, 2: 8, 3: 12} # Adjusted for borders

# Calculate width for each edge in the graph
edge_widths = []
for u, v, _ in graph_from_net.edges:
    # Find the phase of the *downstream* node (approximate line phase)
    # We use 'to_bus' phase as a proxy for the line
    try:
        phase_str = sample_power.loc[v, 'phase']
        phase_count = get_phase_count(phase_str)
        width = phase_count_to_width[phase_count]
    except KeyError:
        width = 1.0
    edge_widths.append(width)

# --- 2. Prepare Edge Colors (R and X values) ---
# Map line indices to graph edges
# Note: pandapower graph edges are usually (from_bus, to_bus, key) or just (u, v)
# We iterate through net.line to match R/X values to specific edges
r_values = []
x_values = []

# Create a map of (u, v) -> index to ensure order
edge_to_idx = {edge[0:2]: i for i, edge in enumerate(graph_from_net.edges)}
r_ordered = [0.0] * len(graph_from_net.edges)
x_ordered = [0.0] * len(graph_from_net.edges)

for idx, row in net.line.iterrows():
    u, v = row['from_bus'], row['to_bus']
    
    # Check if edge exists in graph (it should)
    if graph_from_net.has_edge(u, v):
        # Get the integer index of this edge in the graph list
        # Note: graph edges might be (u, v) or (v, u) since it's undirected
        edge_key = (u, v) if (u, v) in edge_to_idx else (v, u)
        
        if edge_key in edge_to_idx:
            list_idx = edge_to_idx[edge_key]
            
            # Get R/X from sample
            try:
                r_ordered[list_idx] = sample_imp.loc[idx, 'R1_ohm_per_km']
                x_ordered[list_idx] = sample_imp.loc[idx, 'X1_ohm_per_km']
            except KeyError:
                pass

# --- 3. Plotting ---
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Plot R1
ax1 = axes[0]
nx.draw_networkx_nodes(graph_from_net, pos=bus_geo_data, ax=ax1, node_size=0, node_color='black')
edges1 = nx.draw_networkx_edges(
    graph_from_net, pos=bus_geo_data, ax=ax1,
    edge_color=r_ordered, width=edge_widths,
    edge_cmap=plt.cm.YlOrRd
)
cbar1 = plt.colorbar(edges1, ax=ax1, shrink=0.8)
cbar1.set_label(r'Resistance ($\Omega$/km)', fontsize=20)
cbar1.ax.tick_params(labelsize=18) # Adjust labelsize as desired
#ax1.set_title("Line Resistance (R1)")
ax1.axis('off')

# Plot X1
ax2 = axes[1]
nx.draw_networkx_nodes(graph_from_net, pos=bus_geo_data, ax=ax2, node_size=0, node_color='black')
edges2 = nx.draw_networkx_edges(
    graph_from_net, pos=bus_geo_data, ax=ax2,
    edge_color=x_ordered, width=edge_widths,
    edge_cmap=plt.cm.GnBu
)
cbar2 = plt.colorbar(edges2, ax=ax2, shrink=0.8)
cbar2.set_label(r'Reactance ($\Omega$/km)', fontsize=20)
cbar2.ax.tick_params(labelsize=18) # Adjust labelsize as desired
#ax2.set_title("Line Reactance (X1)")
ax2.axis('off')

plt.tight_layout()
plt.show()